# 10.2 Passive reinforcement learning: practice notebook

Learn how good a fixed policy is from experience alone. Average rewards to go, learn a model and solve it, or update after every step with temporal difference learning.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Direct estimation and a learned model from three trials

In [ ]:
# The three trials of AIMA section 21.2 in the 4 x 3 world. Each step costs -0.04, the end square gives +1 or -1.
trials = [["11", "12", "13", "12", "13", "23", "33", "43"],
          ["11", "12", "13", "23", "33", "32", "33", "43"],
          ["11", "21", "31", "32", "42"]]
end = {"43": 1.0, "42": -1.0}

# Direct utility estimation: average the reward to go of every visit.
total, count = {}, {}
for t in trials:
    for i in range(len(t) - 1):
        to_go = -0.04 * (len(t) - 1 - i) + end[t[-1]]
        s = t[i]
        total[s] = total.get(s, 0) + to_go
        count[s] = count.get(s, 0) + 1
print("direct estimate of U(1,1):", round(total["11"] / count["11"], 4))

# Adaptive dynamic programming learns the model first: count where each square led.
moves = {}
for t in trials:
    for i in range(len(t) - 1):
        moves.setdefault(t[i], []).append(t[i + 1])
for s in ["13", "33"]:
    nxt = moves[s]
    print("from", s, ":", {n: str(nxt.count(n)) + "/" + str(len(nxt)) for n in sorted(set(nxt))})

### TD(0) on the five state random walk

In [ ]:
# TD(0) on the five state random walk of Sutton and Barto (example 6.2).
# States A to E, start at C, move left or right with chance 1/2. Leaving on the right gives 1, on the left 0.
import random
rng = random.Random(3)
names = ["A", "B", "C", "D", "E"]
V = {s: 0.5 for s in names}
V["left"], V["right"] = 0.0, 0.0
alpha = 0.1

for episode in range(6):
    i = 2
    while 0 <= i <= 4:
        j = i + (1 if rng.random() < 0.5 else -1)
        nxt = "left" if j < 0 else "right" if j > 4 else names[j]
        reward = 1.0 if nxt == "right" else 0.0
        V[names[i]] = V[names[i]] + alpha * (reward + V[nxt] - V[names[i]])   # TD(0) update
        i = j
print("after 6 episodes:", {s: round(V[s], 3) for s in names})
print("true values:      ", {s: round((k + 1) / 6, 3) for k, s in enumerate(names)})

## Your turn

### Task 1: Direct utility estimation

Write `direct(trials, end, step)` that returns a dictionary from each visited non end square to the average of its rewards to go. Every step from a square costs step, and the last square gives end[last]. Count every visit.

Hint: The square at position i of a trial has len(t) − 1 − i steps before the end.

In [ ]:
TRIALS = [["11", "12", "13", "12", "13", "23", "33", "43"],
          ["11", "12", "13", "23", "33", "32", "33", "43"],
          ["11", "21", "31", "32", "42"]]
END = {"43": 1.0, "42": -1.0}

def direct(trials, end, step):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("round(direct(TRIALS, END, -0.04)['11'], 4)", 0.0933), ("round(direct(TRIALS, END, -0.04)['33'], 4)", 0.9333), ("round(direct(TRIALS, END, -0.04)['21'], 2)", -1.12), ('sorted(direct(TRIALS, END, -0.04))', ['11', '12', '13', '21', '23', '31', '32', '33'])])

### Task 2: One TD update

Write `td_update(u_s, r, u_next, alpha, gamma)` that returns the new U(s) after one TD(0) update.

Hint: Compute the TD error first.

In [ ]:
def td_update(u_s, r, u_next, alpha, gamma):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('round(td_update(0.5, -0.04, 0.8, 0.1, 1.0), 4)', 0.526), ('round(td_update(0.5, 0.0, 0.6, 0.5, 0.9), 4)', 0.52), ('td_update(1.0, 0.0, 1.0, 0.3, 1.0)', 1.0)])

### Task 3: TD(0) on the random walk

Write `td_walk(episodes, alpha, seed)` for the five state random walk (start at C, step left or right with chance 1/2 using `rng.random() < 0.5` for right, reward 1 only when leaving on the right, γ = 1, all values start at 0.5). Return the dictionary of values.

Hint: The target is 0 when leaving on the left, 1 when leaving on the right, and V of the next state otherwise.

In [ ]:
import random

def td_walk(episodes, alpha, seed):
    rng = random.Random(seed)
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("all(abs(td_walk(2000, 0.02, 1)[s] - (k + 1) / 6) < 0.1 for k, s in enumerate('ABCDE'))", True), ("all(abs(td_walk(2000, 0.02, 9)[s] - (k + 1) / 6) < 0.1 for k, s in enumerate('ABCDE'))", True)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Direct utility estimation

(3,3) is visited 3 times with rewards to go 0.96, 0.96 and 0.88.

In [ ]:
TRIALS = [["11", "12", "13", "12", "13", "23", "33", "43"],
          ["11", "12", "13", "23", "33", "32", "33", "43"],
          ["11", "21", "31", "32", "42"]]
END = {"43": 1.0, "42": -1.0}

def direct(trials, end, step):
    total = {}
    count = {}
    for t in trials:
        for i in range(len(t) - 1):
            to_go = step * (len(t) - 1 - i) + end[t[-1]]
            s = t[i]
            total[s] = total.get(s, 0) + to_go
            count[s] = count.get(s, 0) + 1
    result = {}
    for s in total:
        result[s] = total[s] / count[s]
    return result

In [ ]:
check([("round(direct(TRIALS, END, -0.04)['11'], 4)", 0.0933), ("round(direct(TRIALS, END, -0.04)['33'], 4)", 0.9333), ("round(direct(TRIALS, END, -0.04)['21'], 2)", -1.12), ('sorted(direct(TRIALS, END, -0.04))', ['11', '12', '13', '21', '23', '31', '32', '33'])])

### Solution to task 2: One TD update

Test 3: the target equals the guess, so nothing changes.

In [ ]:
def td_update(u_s, r, u_next, alpha, gamma):
    error = r + gamma * u_next - u_s
    return u_s + alpha * error

In [ ]:
check([('round(td_update(0.5, -0.04, 0.8, 0.1, 1.0), 4)', 0.526), ('round(td_update(0.5, 0.0, 0.6, 0.5, 0.9), 4)', 0.52), ('td_update(1.0, 0.0, 1.0, 0.3, 1.0)', 1.0)])

### Solution to task 3: TD(0) on the random walk

With a small α the values settle near 1/6, 2/6, 3/6, 4/6 and 5/6.

In [ ]:
import random

def td_walk(episodes, alpha, seed):
    rng = random.Random(seed)
    names = ["A", "B", "C", "D", "E"]
    V = {}
    for s in names:
        V[s] = 0.5
    for e in range(episodes):
        i = 2
        while 0 <= i <= 4:
            if rng.random() < 0.5:
                j = i + 1
            else:
                j = i - 1
            if j < 0:
                target = 0.0
            elif j > 4:
                target = 1.0
            else:
                target = V[names[j]]
            V[names[i]] = V[names[i]] + alpha * (target - V[names[i]])
            i = j
    return V

In [ ]:
check([("all(abs(td_walk(2000, 0.02, 1)[s] - (k + 1) / 6) < 0.1 for k, s in enumerate('ABCDE'))", True), ("all(abs(td_walk(2000, 0.02, 9)[s] - (k + 1) / 6) < 0.1 for k, s in enumerate('ABCDE'))", True)])